# Phase 5 — p5_05 Figures and Academic Reporting

Doel: op basis van bestaande Phase 5 outputs academisch sterke, leesbare en onderzoek-relevante visualisaties maken, inclusief formele rapportering.


## Methodologische context
- We hergebruiken uitsluitend reeds berekende Phase 5 artefacts (geen nieuwe modeltuning).
- Focus: primaire modellen (`P4_ridge_a1_0`, `F3_rf_90_d12_l2`) met benchmarkcontrast (`P0`, `F0`).
- Rapportering combineert intuitieve duiding (visueel) en academische strengheid (stabiliteit, CI, permutation p-values, trade-offs).


In [1]:

from pathlib import Path
import sys
import os


def _add_project_root_to_syspath() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src").exists() and (candidate / "data_processed").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError("Project root with `src/` and `data_processed/` not found in current path hierarchy.")


PROJECT_ROOT = _add_project_root_to_syspath()
os.environ["MPLCONFIGDIR"] = str((PROJECT_ROOT / ".matplotlib_local").resolve())

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 220
plt.rcParams["axes.titleweight"] = "bold"

PHASE5_DIR = PROJECT_ROOT / "data_results" / "phase5"
FIG_DIR = PHASE5_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_RUNS = ["P4_ridge_a1_0", "F3_rf_90_d12_l2"]
RUN_LABEL = {
    "P4_ridge_a1_0": "Policy Primary (P4 Ridge)",
    "F3_rf_90_d12_l2": "Forecast Primary (F3 RF)",
    "P0_profile_baseline": "Policy Baseline (P0)",
    "F0_persistence_baseline": "Forecast Baseline (F0)",
}
RUN_COLOR = {
    "P4_ridge_a1_0": "#1f77b4",
    "F3_rf_90_d12_l2": "#d62728",
    "P0_profile_baseline": "#8c8c8c",
    "F0_persistence_baseline": "#5f5f5f",
}


def feature_block(feature_name: str) -> str:
    if str(feature_name).startswith("t_"):
        return "T (Time)"
    if str(feature_name).startswith("s_"):
        return "S (Spatial)"
    if str(feature_name).startswith("e_"):
        return "E (External)"
    if str(feature_name).startswith("l_"):
        return "L (Lag)"
    return "Other"


def savefig(fig, filename: str):
    path = FIG_DIR / filename
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


# Load phase5 artifacts
scope = pd.read_csv(PHASE5_DIR / "phase5_model_scope.csv")
global_imp = pd.read_csv(PHASE5_DIR / "phase5_shap_global_importance.csv")
tier_imp = pd.read_csv(PHASE5_DIR / "phase5_shap_tier_importance.csv")
tier_delta = pd.read_csv(PHASE5_DIR / "phase5_shap_tier_delta_tests.csv")
fold_stability = pd.read_csv(PHASE5_DIR / "phase5_shap_fold_stability.csv")
context_metrics = pd.read_csv(PHASE5_DIR / "phase5_context_metrics.csv")
baseline_diag = pd.read_csv(PHASE5_DIR / "phase5_baseline_diagnostics.csv")
iter_log = pd.read_csv(PHASE5_DIR / "phase5_iterative_critique_log.csv")
claims = pd.read_csv(PHASE5_DIR / "phase5_interpretation_claims.csv")
additivity = pd.read_csv(PHASE5_DIR / "phase5_shap_additivity_checks.csv")
rowlevel = pd.read_parquet(PHASE5_DIR / "phase5_shap_rowlevel_samples.parquet")

holdout_results = pd.read_csv(PROJECT_ROOT / "data_results" / "phase4" / "phase4_holdout_results_2025.csv")
policy_holdout = pd.read_parquet(PROJECT_ROOT / "data_results" / "phase4" / "internal" / "full_train_holdout" / "policy_holdout.parquet")
forecast_holdout = pd.read_parquet(PROJECT_ROOT / "data_results" / "phase4" / "internal" / "full_train_holdout" / "forecast_holdout.parquet")

# Helper views
g_hold = global_imp.loc[global_imp["context"].eq("holdout")].copy()

print("Loaded Phase 5 artefacts:")
print("global_imp", global_imp.shape, "tier_delta", tier_delta.shape, "rowlevel", rowlevel.shape)
print("Figures dir:", FIG_DIR)

display(scope)


Loaded Phase 5 artefacts:
global_imp (696, 9) tier_delta (184, 11) rowlevel (116596, 106)
Figures dir: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_results/phase5/figures


,track,role,run_id,model_family,feature_set_id,row_filter,claim_scope
0,policy,primary,P4_ridge_a1_0,ridge,policy_tse_exante_strict,none,fold_plus_holdout
1,forecast,primary,F3_rf_90_d12_l2,random_forest,forecast_tsel_core_strict_lagvalid,l_valid_all_core==1,fold_plus_holdout
2,policy,benchmark,P0_profile_baseline,baseline,baseline_profile,none,benchmark_contrast
3,forecast,benchmark,F0_persistence_baseline,baseline,baseline_persistence,none,benchmark_contrast


## Figuur 1-2: Globale SHAP-dominantie en blokbijdragen

Intuitief: welke features domineren per primair model?
Academisch: welk deel van de verklaringsmassa komt uit T/S/E/L-blokken?


In [2]:

# Figure 1: top-15 holdout SHAP features per primary run
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, run_id in zip(axes, PRIMARY_RUNS):
    sub = g_hold.loc[g_hold["run_id"].eq(run_id)].sort_values("rank").head(15).copy()
    sub = sub.iloc[::-1]
    sns.barplot(data=sub, x="mean_abs_shap", y="feature_name", color=RUN_COLOR[run_id], ax=ax)
    ax.set_title(f"Top-15 SHAP Features ({RUN_LABEL[run_id]})")
    ax.set_xlabel("Mean |SHAP| (holdout)")
    ax.set_ylabel("Feature")
fig.suptitle("Figure 1 — Holdout SHAP Dominance (Primary Models)", fontsize=18, fontweight="bold")
fig1_path = savefig(fig, "fig01_holdout_top15_shap_primary.png")

# Figure 2: feature-block contribution share
block_df = g_hold.loc[g_hold["run_id"].isin(PRIMARY_RUNS)].copy()
block_df["block"] = block_df["feature_name"].map(feature_block)
block_share = (
    block_df.groupby(["run_id", "block"], as_index=False)["mean_abs_shap"].sum()
)
block_share["share"] = block_share["mean_abs_shap"] / block_share.groupby("run_id")["mean_abs_shap"].transform("sum")
block_pivot = block_share.pivot(index="run_id", columns="block", values="share").fillna(0)
for col in ["T (Time)", "S (Spatial)", "E (External)", "L (Lag)"]:
    if col not in block_pivot.columns:
        block_pivot[col] = 0.0
block_pivot = block_pivot[["T (Time)", "S (Spatial)", "E (External)", "L (Lag)"]]
block_pivot.index = [RUN_LABEL[x] for x in block_pivot.index]

fig, ax = plt.subplots(figsize=(13, 7))
block_pivot.plot(kind="bar", stacked=True, ax=ax, color=["#4C78A8", "#72B7B2", "#F58518", "#E45756"])
ax.set_title("Figure 2 — Relative SHAP Mass by Feature Family (Holdout)", fontweight="bold")
ax.set_ylabel("Share of total mean |SHAP|")
ax.set_xlabel("")
ax.legend(title="Feature Family", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_ylim(0, 1)
fig2_path = savefig(fig, "fig02_shap_block_contribution_share.png")

display(Markdown(f"Saved: `{fig1_path.name}`, `{fig2_path.name}`"))
display(block_share.sort_values(["run_id", "share"], ascending=[True, False]).groupby("run_id").head(6))


Saved: `fig01_holdout_top15_shap_primary.png`, `fig02_shap_block_contribution_share.png`

,run_id,block,mean_abs_shap,share
1,F3_rf_90_d12_l2,L (Lag),0.234550,0.843239
3,F3_rf_90_d12_l2,T (Time),0.025290,0.090920
2,F3_rf_90_d12_l2,S (Spatial),0.011448,0.041158
0,F3_rf_90_d12_l2,E (External),0.006866,0.024683
6,P4_ridge_a1_0,T (Time),0.250685,0.358561
5,P4_ridge_a1_0,S (Spatial),0.237618,0.339872
4,P4_ridge_a1_0,E (External),0.210837,0.301567


## Figuur 3-4: Fold-stabiliteit van SHAP-signalen

Intuitief: blijven belangrijke features ook belangrijk over folds?
Academisch: rank-variabiliteit en top-10 frequentie als stabiliteitsmaat.


In [3]:

fold_df = global_imp.loc[global_imp["context"].str.startswith("fold_")].copy()

# Figure 3: heatmap of fold ranks for top holdout features
fig, axes = plt.subplots(1, 2, figsize=(18, 10), sharey=False)
for ax, run_id in zip(axes, PRIMARY_RUNS):
    top_feats = g_hold.loc[g_hold["run_id"].eq(run_id)].sort_values("rank").head(12)["feature_name"].tolist()
    hm = fold_df.loc[(fold_df["run_id"].eq(run_id)) & (fold_df["feature_name"].isin(top_feats))]
    hm = hm.pivot(index="feature_name", columns="fold_id", values="rank").reindex(top_feats)
    sns.heatmap(hm, annot=True, fmt=".0f", cmap="YlGnBu_r", cbar=True, ax=ax)
    ax.set_title(f"Fold Rank Heatmap ({RUN_LABEL[run_id]})")
    ax.set_xlabel("Fold")
    ax.set_ylabel("Feature")
fig.suptitle("Figure 3 — Fold Rank Stability for Top Holdout Features", fontsize=17, fontweight="bold")
fig3_path = savefig(fig, "fig03_fold_rank_heatmap_top_features.png")

# Figure 4: stability scatter (mean rank vs std rank)
fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)
for ax, run_id in zip(axes, PRIMARY_RUNS):
    top_feats = g_hold.loc[g_hold["run_id"].eq(run_id)].sort_values("rank").head(25)["feature_name"].tolist()
    st = fold_stability.loc[(fold_stability["run_id"].eq(run_id)) & (fold_stability["feature_name"].isin(top_feats))].copy()
    sc = ax.scatter(
        st["mean_rank"],
        st["std_rank"],
        c=st["top10_freq"],
        cmap="viridis",
        s=80 + 220 * st["sign_consistency"],
        alpha=0.9,
        edgecolor="black",
        linewidth=0.3,
    )
    for _, r in st.nsmallest(8, "mean_rank").iterrows():
        ax.text(r["mean_rank"] + 0.2, r["std_rank"] + 0.05, r["feature_name"], fontsize=9)
    ax.set_title(f"{RUN_LABEL[run_id]}")
    ax.set_xlabel("Mean Fold Rank (lower = more important)")
    ax.set_ylabel("Std Fold Rank")
cbar = fig.colorbar(sc, ax=axes.ravel().tolist(), shrink=0.9)
cbar.set_label("Top-10 frequency")
fig.suptitle("Figure 4 — SHAP Stability Landscape", fontsize=17, fontweight="bold")
fig4_path = savefig(fig, "fig04_shap_stability_scatter.png")

display(Markdown(f"Saved: `{fig3_path.name}`, `{fig4_path.name}`"))


Saved: `fig03_fold_rank_heatmap_top_features.png`, `fig04_shap_stability_scatter.png`

## Figuur 5: Per-tier effectverschillen met onzekerheidskwantificatie

Intuitief: voor welke features verschilt impact tussen centrum en vesten/rand?
Academisch: forest-plot met 95% CI en significantie op basis van permutation + CI-regel.


In [4]:

fig, axes = plt.subplots(1, 2, figsize=(20, 9), sharex=False)

for ax, run_id in zip(axes, PRIMARY_RUNS):
    sub = tier_delta.loc[tier_delta["run_id"].eq(run_id)].copy()
    sub["abs_delta"] = sub["delta_mean_abs_shap"].abs()
    sub = sub.sort_values("abs_delta", ascending=False).head(15)
    sub = sub.sort_values("delta_mean_abs_shap")

    y = np.arange(len(sub))
    colors = np.where(sub["significant"], "#2ca02c", "#7f7f7f")
    ax.hlines(y=y, xmin=sub["ci_low"], xmax=sub["ci_high"], color=colors, lw=2)
    ax.scatter(sub["delta_mean_abs_shap"], y, c=colors, s=70, zorder=3)
    ax.axvline(0.0, color="black", lw=1, ls="--")
    ax.set_yticks(y)
    ax.set_yticklabels(sub["feature_name"])
    ax.set_title(f"{RUN_LABEL[run_id]}")
    ax.set_xlabel("Delta mean |SHAP| (centrum - vesten/rand)")

fig.suptitle("Figure 5 — Tier Delta Forest Plot (Holdout)", fontsize=17, fontweight="bold")
fig5_path = savefig(fig, "fig05_tier_delta_forest.png")

display(Markdown(f"Saved: `{fig5_path.name}`"))


Saved: `fig05_tier_delta_forest.png`

## Figuur 6: Intuitieve SHAP-dependence zoom op topfeatures

Intuitief: hoe verandert SHAP-bijdrage over feature-waarde?
Academisch: dit is exploratief (associatief), niet causaal.


In [5]:

# Pick 3 interpretable high-rank features per model for dependence plots
run_feature_candidates = {
    "P4_ridge_a1_0": [
        "t_hour_cos",
        "t_hour_sin",
        "e_n_concurrent_events_capped",
        "e_event_scale_max",
        "s_log_total_capacity",
    ],
    "F3_rf_90_d12_l2": [
        "l_occ_lag_1h",
        "l_occ_lag_24h",
        "l_occ_lag_168h",
        "l_occ_lag_delta_1h_24h",
        "t_hour_sin",
    ],
}

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

for ridx, run_id in enumerate(PRIMARY_RUNS):
    hold_rl = rowlevel.loc[(rowlevel["run_id"].eq(run_id)) & (rowlevel["context"].eq("holdout"))].copy()
    hold_rl["rounded_hour"] = pd.to_datetime(hold_rl["rounded_hour"], errors="coerce")

    raw_hold = policy_holdout.copy() if run_id.startswith("P") else forecast_holdout.copy()
    raw_hold["rounded_hour"] = pd.to_datetime(raw_hold["rounded_hour"], errors="coerce")

    chosen = []
    for feat in run_feature_candidates[run_id]:
        if feat in raw_hold.columns and f"shap__{feat}" in hold_rl.columns:
            chosen.append(feat)
        if len(chosen) == 3:
            break

    for cidx in range(3):
        ax = axes[ridx, cidx]
        if cidx >= len(chosen):
            ax.axis("off")
            continue

        feat = chosen[cidx]
        tmp = hold_rl[["parking_id", "rounded_hour", f"shap__{feat}"]].merge(
            raw_hold[["parking_id", "rounded_hour", feat]],
            on=["parking_id", "rounded_hour"],
            how="left",
        )
        tmp = tmp.dropna(subset=[feat, f"shap__{feat}"]).copy()

        ax.hexbin(
            tmp[feat].to_numpy(dtype=float),
            tmp[f"shap__{feat}"].to_numpy(dtype=float),
            gridsize=42,
            cmap="viridis",
            mincnt=1,
        )
        ax.axhline(0.0, color="black", lw=1, ls="--")
        ax.set_title(f"{RUN_LABEL[run_id]} — {feat}")
        ax.set_xlabel(feat)
        ax.set_ylabel(f"SHAP({feat})")

fig.suptitle("Figure 6 — SHAP Dependence (Hexbin) on Selected Features", fontsize=17, fontweight="bold")
fig6_path = savefig(fig, "fig06_shap_dependence_zoom.png")

display(Markdown(f"Saved: `{fig6_path.name}`"))


Saved: `fig06_shap_dependence_zoom.png`

## Figuur 7-8: Prestatie- en benchmarkcontext

Intuitief: hoe verhouden primary en baseline zich?
Academisch: expliciete trade-off rapportering (nauwkeurigheid vs methodologische keuze).


In [6]:

# Figure 7: holdout MAE/RMSE comparison primary vs baseline
pairs = pd.DataFrame(
    [
        {"track": "policy", "primary": "P4_ridge_a1_0", "baseline": "P0_profile_baseline"},
        {"track": "forecast", "primary": "F3_rf_90_d12_l2", "baseline": "F0_persistence_baseline"},
    ]
)

rows = []
for _, r in pairs.iterrows():
    p_row = holdout_results.loc[holdout_results["run_id"].eq(r["primary"])].iloc[0]
    b_row = holdout_results.loc[holdout_results["run_id"].eq(r["baseline"])].iloc[0]
    rows.extend(
        [
            {"track": r["track"], "model": "Primary", "run_id": r["primary"], "mae": float(p_row["mae"]), "rmse": float(p_row["rmse"])},
            {"track": r["track"], "model": "Baseline", "run_id": r["baseline"], "mae": float(b_row["mae"]), "rmse": float(b_row["rmse"])},
        ]
    )
hold_cmp = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)
sns.barplot(data=hold_cmp, x="track", y="mae", hue="model", ax=axes[0], palette=["#4C78A8", "#9D9D9D"])
axes[0].set_title("Holdout MAE: Primary vs Baseline")
axes[0].set_xlabel("")
axes[0].set_ylabel("MAE")

sns.barplot(data=hold_cmp, x="track", y="rmse", hue="model", ax=axes[1], palette=["#4C78A8", "#9D9D9D"])
axes[1].set_title("Holdout RMSE: Primary vs Baseline")
axes[1].set_xlabel("")
axes[1].set_ylabel("RMSE")
for ax in axes:
    ax.legend(title="")
fig.suptitle("Figure 7 — Accuracy Trade-off Context", fontsize=17, fontweight="bold")
fig7_path = savefig(fig, "fig07_primary_vs_baseline_holdout_accuracy.png")

# Figure 8: baseline stage shares on holdout
b_hold = baseline_diag.loc[baseline_diag["context"].eq("holdout")].copy()

p0 = b_hold.loc[b_hold["run_id"].eq("P0_profile_baseline")].iloc[0]
f0 = b_hold.loc[b_hold["run_id"].eq("F0_persistence_baseline")].iloc[0]

p0_stages = pd.Series(
    {
        "parking_hour_weekday": p0.get("share_stage_parking_hour_weekday", np.nan),
        "parking_hour": p0.get("share_stage_parking_hour", np.nan),
        "parking": p0.get("share_stage_parking", np.nan),
        "global": p0.get("share_stage_global", np.nan),
    }
)
f0_stages = pd.Series(
    {
        "lag1": f0.get("share_stage_lag1", np.nan),
        "lag24": f0.get("share_stage_lag24", np.nan),
        "fallback_profile": f0.get("share_stage_fallback_profile", np.nan),
    }
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar(p0_stages.index, p0_stages.values, color=["#1f77b4", "#4c78a8", "#9ecae9", "#c6dbef"])
axes[0].set_title("P0 Baseline Stage Usage (Holdout)")
axes[0].set_ylabel("Share")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(f0_stages.index, f0_stages.values, color=["#d62728", "#ff9896", "#c7c7c7"])
axes[1].set_title("F0 Baseline Stage Usage (Holdout)")
axes[1].set_ylabel("Share")
axes[1].tick_params(axis="x", rotation=25)

fig.suptitle("Figure 8 — Baseline Mechanism Decomposition", fontsize=17, fontweight="bold")
fig8_path = savefig(fig, "fig08_baseline_stage_decomposition.png")

display(Markdown(f"Saved: `{fig7_path.name}`, `{fig8_path.name}`"))
display(hold_cmp)


Saved: `fig07_primary_vs_baseline_holdout_accuracy.png`, `fig08_baseline_stage_decomposition.png`

,track,model,run_id,mae,rmse
0,policy,Primary,P4_ridge_a1_0,0.216910,0.265701
1,policy,Baseline,P0_profile_baseline,0.206404,0.256322
2,forecast,Primary,F3_rf_90_d12_l2,0.064971,0.099542
3,forecast,Baseline,F0_persistence_baseline,0.063474,0.111438


## Figuur 9 + geformaliseerde rapportering

Intuitief: zijn SHAP-decomposities numeriek consistent met modeloutput?
Academisch: additiviteitsdiagnostiek + geautomatiseerde rapporttekst met hoofdbevindingen.


In [7]:

# Figure 9: additivity diagnostics across contexts
add = additivity.copy()
add["run_label"] = add["run_id"].map(RUN_LABEL)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
sns.barplot(data=add, x="context", y="mean_abs_error", hue="run_label", ax=axes[0])
axes[0].set_title("Mean Additivity Absolute Error")
axes[0].set_xlabel("Context")
axes[0].set_ylabel("|base + sum(SHAP) - prediction|")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(title="", fontsize=9)

sns.barplot(data=add, x="context", y="p95_abs_error", hue="run_label", ax=axes[1])
axes[1].set_title("P95 Additivity Absolute Error")
axes[1].set_xlabel("Context")
axes[1].set_ylabel("P95 |error|")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend(title="", fontsize=9)

fig.suptitle("Figure 9 — SHAP Additivity Diagnostics", fontsize=17, fontweight="bold")
fig9_path = savefig(fig, "fig09_shap_additivity_diagnostics.png")

# Build formal report markdown
sig_share = (
    tier_delta.groupby("run_id", as_index=False)["significant"].mean().rename(columns={"significant": "sig_share"})
)
sig_map = {r["run_id"]: float(r["sig_share"]) for _, r in sig_share.iterrows()}

jaccard_row = iter_log.loc[iter_log["rule"].eq("top10_jaccard_threshold")].iloc[0]
tradeoff_row = iter_log.loc[iter_log["rule"].eq("benchmark_better_than_primary_holdout_mae")].iloc[0]

p4_top10 = g_hold.loc[g_hold["run_id"].eq("P4_ridge_a1_0")].sort_values("rank").head(10)["feature_name"].tolist()
f3_top10 = g_hold.loc[g_hold["run_id"].eq("F3_rf_90_d12_l2")].sort_values("rank").head(10)["feature_name"].tolist()

p0_mae = float(holdout_results.loc[holdout_results["run_id"].eq("P0_profile_baseline"), "mae"].iloc[0])
p4_mae = float(holdout_results.loc[holdout_results["run_id"].eq("P4_ridge_a1_0"), "mae"].iloc[0])
f0_mae = float(holdout_results.loc[holdout_results["run_id"].eq("F0_persistence_baseline"), "mae"].iloc[0])
f3_mae = float(holdout_results.loc[holdout_results["run_id"].eq("F3_rf_90_d12_l2"), "mae"].iloc[0])

policy_gap = p4_mae - p0_mae
forecast_gap = f3_mae - f0_mae

report_lines = [
    "# Phase 5 Figure Report (Intuitive + Academic)",
    "",
    "Datum: 2026-03-14",
    "",
    "## 1. Scope",
    "- Primary interpretatiemodellen: P4 (policy) en F3 (forecast).",
    "- Benchmarkcontrast: P0 en F0.",
    "- Basis: bestaande Phase 5 artefacts; geen nieuwe tuning.",
    "",
    "## 2. Figuuroverzicht",
    "- Figure 1: holdout top-15 SHAP features per primary model.",
    "- Figure 2: relatieve SHAP-massa per featurefamilie (T/S/E/L).",
    "- Figure 3: fold-rank heatmaps voor top holdout features.",
    "- Figure 4: stabiliteitslandschap (mean rank, std rank, top10 frequency).",
    "- Figure 5: tier-delta forest plots met 95% CI en significantie.",
    "- Figure 6: dependence-zoom voor geselecteerde topfeatures.",
    "- Figure 7: holdout accuracy trade-off (primary vs baseline).",
    "- Figure 8: baseline-stage decompositie op holdout.",
    "- Figure 9: additiviteitsdiagnostiek van SHAP.",
    "",
    "## 3. Intuitieve kernbevindingen",
    f"- Policy P4 top-10 wordt gedomineerd door: {', '.join(p4_top10)}.",
    f"- Forecast F3 top-10 wordt gedomineerd door: {', '.join(f3_top10)}.",
    f"- Policy holdout MAE-gap (P4 - P0): {policy_gap:.4f} (P0 beter).",
    f"- Forecast holdout MAE-gap (F3 - F0): {forecast_gap:.4f} (F0 beter op full holdout).",
    "",
    "## 4. Academische interpretatie",
    "### 4.1 Stabiliteit",
    f"- Iteratieve stabiliteitsregel activeerde rerun: top10_jaccard={float(jaccard_row['value']):.4f} < 0.60.",
    "- Dit ondersteunt methodologische strengheid: claims steunen op gehercalculeerde SHAP-structuur.",
    "",
    "### 4.2 Tier-heterogeniteit",
    f"- Significantie-aandeel tier-deltas: P4={sig_map.get('P4_ridge_a1_0', float('nan'))*100:.1f}%, F3={sig_map.get('F3_rf_90_d12_l2', float('nan'))*100:.1f}%.",
    "- Conclusie: forecast vertoont sterkere, consistenter meetbare tierverschillen dan policy in deze opzet.",
    "",
    "### 4.3 Trade-off versus benchmark",
    f"- Benchmark-beter-regel geactiveerd: {bool(tradeoff_row['triggered'])}.",
    "- Thesisclaim moet expliciet scheiden: predictieve minimumfout vs methodologisch interpreteerbare modelkeuze.",
    "",
    "### 4.4 Additiviteitscontrole",
    "- Figure 9 toont lage additiviteitsfouten over contexten, wat de numerieke consistentie van SHAP-decomposities ondersteunt.",
    "",
    "## 5. Beperkingen",
    "- SHAP-interpretatie blijft associatief en niet causaal.",
    "- Per-tier conclusies moeten beperkt blijven tot features met CI- en permutation-ondersteuning.",
    "- Benchmarksuperioriteit op holdout-MAE blijft een expliciete randvoorwaarde in de discussie.",
    "",
    "## 6. Praktische thesis-aanbeveling",
    "- Gebruik Figure 1-2 om globale verhaalstructuur te introduceren.",
    "- Gebruik Figure 3-5 voor methodologische robuustheid en heterogeniteit.",
    "- Gebruik Figure 7-8 om het primaire trade-off argument expliciet te maken.",
    "- Verwijs Figure 6 als intuitieve verdieping; positioneer dit als exploratief.",
]

report_path = PHASE5_DIR / "phase5_figures_report.md"
report_path.write_text("\n".join(report_lines))

# quick dashboard table
fig_index = pd.DataFrame(
    {
        "figure": [f"Figure {i}" for i in range(1, 10)],
        "file": [
            "fig01_holdout_top15_shap_primary.png",
            "fig02_shap_block_contribution_share.png",
            "fig03_fold_rank_heatmap_top_features.png",
            "fig04_shap_stability_scatter.png",
            "fig05_tier_delta_forest.png",
            "fig06_shap_dependence_zoom.png",
            "fig07_primary_vs_baseline_holdout_accuracy.png",
            "fig08_baseline_stage_decomposition.png",
            "fig09_shap_additivity_diagnostics.png",
        ],
        "purpose": [
            "global dominance",
            "family-level contribution",
            "fold-rank consistency",
            "stability summary",
            "tier heterogeneity + uncertainty",
            "dependence zoom",
            "accuracy trade-off",
            "baseline mechanism",
            "SHAP numeric validity",
        ],
    }
)

print("Saved report:", report_path)
display(fig_index)
display(Markdown(f"Saved: `{fig9_path.name}`"))


Saved report: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_results/phase5/phase5_figures_report.md


,figure,file,purpose
0,Figure 1,fig01_holdout_top15_shap_primary.png,global dominance
1,Figure 2,fig02_shap_block_contribution_share.png,family-level contribution
2,Figure 3,fig03_fold_rank_heatmap_top_features.png,fold-rank consistency
3,Figure 4,fig04_shap_stability_scatter.png,stability summary
4,Figure 5,fig05_tier_delta_forest.png,tier heterogeneity + uncertainty
5,Figure 6,fig06_shap_dependence_zoom.png,dependence zoom
6,Figure 7,fig07_primary_vs_baseline_holdout_accuracy.png,accuracy trade-off
7,Figure 8,fig08_baseline_stage_decomposition.png,baseline mechanism
8,Figure 9,fig09_shap_additivity_diagnostics.png,SHAP numeric validity


Saved: `fig09_shap_additivity_diagnostics.png`

## Slot

Deze notebook levert zowel thesis-klare visualisaties als formele rapportering.
Gebruik `phase5_figures_report.md` als begeleidende tekst bij de figuren in je resultatenhoofdstuk.
